# CatBoost with Original DHS Features
This notebook loads `rural_cleaned.pkl` and trains CatBoost using selected original categorical features.

**Important:** Review the predictor and leakage columns before final research use.

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from catboost import CatBoostClassifier


In [ ]:
df = pd.read_pickle('../data/interim/rural_cleaned.pkl')

print('Dataset shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
df.head()


In [ ]:
print('Data types:')
print(df.dtypes)

print('\nMissing values:')
print(df.isnull().sum())


In [ ]:
target = 'anemia_binary'

print(df[target].value_counts())
print('\nPercentage:')
print(df[target].value_counts(normalize=True) * 100)


In [ ]:
# IMPORTANT: Use the same predictor variables defined in your research methodology.
# Do NOT automatically use every column in rural_cleaned.pkl.
# Update this list if your existing feature-engineering notebook used a different set.

predictor_cols = [
    'v190a', 'v106', 'v133', 'm14_1', 'v113', 'v116',
    'v012', 'v013', 'v024', 'v025', 'v151', 'v136', 'v201'
]

# Keep only predictors that actually exist in the dataset
predictor_cols = [col for col in predictor_cols if col in df.columns]

print('Predictors being used:')
print(predictor_cols)

X = df[predictor_cols].copy()
y = df[target].copy()

print('\nX shape:', X.shape)
print('y shape:', y.shape)


In [ ]:
# Safety check: make sure target-related variables are NOT predictors
leakage_cols = ['anemia_binary', 'v456', 'v457', 'anemia_level']
print('Potential leakage columns present in X:', [c for c in leakage_cols if c in X.columns])


In [ ]:
# Categorical DHS variables
# Wealth, education and age group are ordinal, but this experiment treats them as categories.
cat_cols = [
    'v106',   # education level
    'v190a',  # wealth index
    'v113',   # water source
    'v116',   # toilet type
    'v024',   # province
    'v025',   # residence
    'v151',   # ecological region
    'v013'    # age group
]

cat_cols = [col for col in cat_cols if col in X.columns]

print('Categorical columns:')
print(cat_cols)

# CatBoost expects categorical values to be strings and without NaN
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training set:', X_train.shape)
print('Test set:', X_test.shape)


In [ ]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()
scale_pos_weight = negative / positive

print('Negative cases:', negative)
print('Positive cases:', positive)
print('Scale positive weight:', scale_pos_weight)


In [ ]:
cat = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function='Logloss',
    eval_metric='AUC',
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=100,
    allow_writing_files=False
)

cat.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    early_stopping_rounds=100,
    verbose=100
)


In [ ]:
pred = cat.predict(X_test).astype(int).flatten()
proba = cat.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
auc = roc_auc_score(y_test, proba)

print('========== CATBOOST RESULTS ==========')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1 Score : {f1:.4f}')
print(f'AUC      : {auc:.4f}')


In [ ]:
print(classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred)
print('Confusion Matrix:')
print(cm)


In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': cat.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance


In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance.head(15).sort_values('importance')

plt.figure(figsize=(10, 6))
plt.barh(top_features['feature'], top_features['importance'])
plt.xlabel('Feature Importance')
plt.ylabel('Features')
plt.title('Top CatBoost Feature Importances')
plt.show()


In [ ]:
results = pd.DataFrame([{
    'model': 'CatBoost Original Features',
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'auc': auc
}])

os.makedirs('../results/metrics', exist_ok=True)
results.to_csv('../results/metrics/catboost_original_features.csv', index=False)

print(results)
print('\nResults saved successfully.')


In [ ]:
os.makedirs('../models/ml', exist_ok=True)

with open('../models/ml/catboost_original_features.pkl', 'wb') as f:
    pickle.dump(cat, f)

print('CatBoost model saved successfully!')
